# Chapter 12: Knowledge Graphs - Implementation

In [ ]:
import numpy as np
from collections import defaultdict
import random

print('Imports complete')

In [ ]:
class KnowledgeGraph:
    def __init__(self):
        self.triples = []
        self.entities = set()
        self.relations = set()
    
    def add_triple(self, h, r, t):
        self.triples.append((h, r, t))
        self.entities.add(h)
        self.entities.add(t)
        self.relations.add(r)
    
    def get_neighbors(self, entity, relation=None):
        neighbors = []
        for h, r, t in self.triples:
            if h == entity and (relation is None or r == relation):
                neighbors.append((r, t))
        return neighbors

print('Knowledge Graph class defined')

In [ ]:
class TransE:
    def __init__(self, kg, dim=50, margin=1.0, lr=0.01):
        self.kg = kg
        self.dim = dim
        self.margin = margin
        self.lr = lr
        
        # Initialize embeddings
        self.entity_emb = {}
        self.relation_emb = {}
        
        for e in kg.entities:
            self.entity_emb[e] = np.random.randn(dim) * 0.01
        for r in kg.relations:
            self.relation_emb[r] = np.random.randn(dim) * 0.01
    
    def distance(self, h, r, t):
        h_vec = self.entity_emb[h]
        r_vec = self.relation_emb[r]
        t_vec = self.entity_emb[t]
        return np.linalg.norm(h_vec + r_vec - t_vec)
    
    def train(self, epochs=100, batch_size=10):
        for epoch in range(epochs):
            loss = 0
            for _ in range(batch_size):
                # Positive triple
                h, r, t = random.choice(self.kg.triples)
                
                # Negative triple (corrupt tail)
                t_neg = random.choice(list(self.kg.entities))
                
                # Compute loss
                pos_dist = self.distance(h, r, t)
                neg_dist = self.distance(h, r, t_neg)
                
                margin_loss = max(0, self.margin + pos_dist - neg_dist)
                loss += margin_loss
                
                if margin_loss > 0:
                    # Gradient update (simplified)
                    h_vec = self.entity_emb[h]
                    r_vec = self.relation_emb[r]
                    t_vec = self.entity_emb[t]
                    t_neg_vec = self.entity_emb[t_neg]
                    
                    # Positive gradient
                    grad = (h_vec + r_vec - t_vec)
                    self.entity_emb[h] -= self.lr * grad
                    self.relation_emb[r] -= self.lr * grad
                    self.entity_emb[t] += self.lr * grad
                    
                    # Negative gradient
                    grad_neg = (h_vec + r_vec - t_neg_vec)
                    self.entity_emb[h] += self.lr * grad_neg
                    self.relation_emb[r] += self.lr * grad_neg
                    self.entity_emb[t_neg] -= self.lr * grad_neg
            
            if epoch % 20 == 0:
                print(f'Epoch {epoch}: loss = {loss:.3f}')
    
    def predict_tail(self, h, r, top_k=5):
        scores = []
        for t in self.kg.entities:
            score = -self.distance(h, r, t)
            scores.append((t, score))
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

print('TransE model implemented')

In [ ]:
print('=== Knowledge Graph Example ===')

kg = KnowledgeGraph()
kg.add_triple('Alice', 'friendOf', 'Bob')
kg.add_triple('Bob', 'friendOf', 'Charlie')
kg.add_triple('Alice', 'worksAt', 'Google')
kg.add_triple('Bob', 'worksAt', 'Facebook')

print(f'Entities: {kg.entities}')
print(f'Relations: {kg.relations}')
print(f'Triples: {len(kg.triples)}')

In [ ]:
print('\n=== Training TransE ===')
model = TransE(kg, dim=10)
model.train(epochs=100, batch_size=5)

print('\n=== Link Prediction ===')
predictions = model.predict_tail('Alice', 'friendOf', top_k=3)
print(f"Predicted friends of Alice: {[p[0] for p in predictions]}")

## Real-World: Movie Recommendations

Building a recommendation system using knowledge graph embeddings.